In [61]:
import kagglehub
import pandas as pd
import numpy as np
import os
import warnings
from google.colab import drive

# Supresión de warnings
# warnings.filterwarnings('ignore')
# pd.options.mode.chained_assignment = None

# Configuración de visualización del DataFrame
pd.set_option('display.max_columns', None)   # muestra todas las columnas
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)


In [62]:
# Montar Google Drive (te va a pedir autorización la primera vez)
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:

# ── Configuración ──────────────────────────────────────────
DATA_PATH_LOCAL         = r"data/raw/archivo.csv"                  # ← CAMBIAR: ruta local
DRIVE_PATH = "/MyDrive/trabajo/ds/huracan/huracan_plantilla_raw.csv"  # ← CAMBIAR: ruta dentro de tu Drive
# ───────────────────────────────────────────────────────────

df_raw = None

# Prioridad 1: archivo local ya descargado
if os.path.exists(DATA_PATH_LOCAL):
    df_raw = pd.read_csv(DATA_PATH_LOCAL)
    print(f"✅ Cargado desde archivo local: {DATA_PATH_LOCAL}")

# Prioridad 2: Google Drive
if df_raw is None:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        ruta_completa = f"/content/drive{DRIVE_PATH}"
        if os.path.exists(ruta_completa):
            df_raw = pd.read_csv(ruta_completa)
            print(f"✅ Cargado desde Google Drive: {ruta_completa}")
            # Guardar localmente para próximas ejecuciones
            os.makedirs(os.path.dirname(DATA_PATH_LOCAL), exist_ok=True)
            df_raw.to_csv(DATA_PATH_LOCAL, index=False)
            print(f"💾 Raw guardado localmente: {DATA_PATH_LOCAL}")
        else:
            print(f"⚠️  Archivo no encontrado en Drive: {ruta_completa}")
    except Exception as e:
        print(f"⚠️  No se pudo montar Drive: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Cargado desde Google Drive: /content/drive/MyDrive/trabajo/ds/huracan/huracan_plantilla_raw.csv
💾 Raw guardado localmente: data/raw/archivo.csv


In [53]:
# ─── Primeras y últimas filas ──────────────────────────────────────────────
print('\n--- Primeras 5 filas ---')
display(df_raw.head())

print('\n--- Últimas 5 filas ---')
display(df_raw.tail())

# ─── Dimensiones ───────────────────────────────────────────────────────────
print(f'Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}')

# ─── Tipos de datos ────────────────────────────────────────────────────────
print('\n--- Info general ---')
df_raw.info()

# ─── Valores nulos por columna ─────────────────────────────────────────────
print('\n--- Nulos por columna ---')
# Verificamos si hay algún nulo en toda la tabla
if df_raw.isnull().sum().sum() == 0:
    print('✅ Sin valores nulos')
else:
    # Sino mostramos tabla filtrada de nulos
    nulos = pd.DataFrame({
        'Nulos': df_raw.isnull().sum(),
        'Porcentaje': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
    })
    display(nulos[nulos['Nulos'] > 0])

# ─── Duplicados ────────────────────────────────────────────────────────────
print(f'\nFilas duplicadas: {df_raw.duplicated().sum()}')

# ─── Estadísticas descriptivas ─────────────────────────────────────────────
print('\n--- Estadísticas descriptivas ---')
display(df_raw.describe())


--- Primeras 5 filas ---


,#,Jugadores,F. Nacim./Edad,Nac.,Valor de mercado
0,32,Sebastián Meza Portero,14/03/2000 (26),NaN,450 mil €
1,NaN,Sebastián Meza,NaN,NaN,NaN
2,NaN,Portero,NaN,NaN,NaN
3,1,Hernán Galíndez Portero,30/03/1987 (39),NaN,300 mil €
4,NaN,Hernán Galíndez,NaN,NaN,NaN



--- Últimas 5 filas ---


,#,Jugadores,F. Nacim./Edad,Nac.,Valor de mercado
85,NaN,Eric Ramírez,NaN,NaN,NaN
86,NaN,Delantero centro,NaN,NaN,NaN
87,18,Luciano Giménez Delantero centro,18/02/2000 (26),NaN,300 mil €
88,NaN,Luciano Giménez,NaN,NaN,NaN
89,NaN,Delantero centro,NaN,NaN,NaN


Filas: 90 | Columnas: 5

--- Info general ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   #                 30 non-null     object 
 1   Jugadores         90 non-null     object 
 2   F. Nacim./Edad    30 non-null     object 
 3   Nac.              0 non-null      float64
 4   Valor de mercado  30 non-null     object 
dtypes: float64(1), object(4)
memory usage: 3.6+ KB

--- Nulos por columna ---


,Nulos,Porcentaje
#,60,66.67
F. Nacim./Edad,60,66.67
Nac.,90,100.00
Valor de mercado,60,66.67



Filas duplicadas: 18

--- Estadísticas descriptivas ---


,Nac.
count,0.00
mean,NaN
std,NaN
min,NaN
25%,NaN
50%,NaN
75%,NaN
max,NaN


In [54]:
# ── Eliminar filas con nulos ───────────────────────────────────────────────

df_raw = df_raw.dropna(subset=['#'])

In [55]:
# Reemplazar '-' por NaN real en todo el dataframe
df_raw = df_raw.replace('-', np.nan)

In [56]:
# Ver estado actual de nulos
nulos = pd.DataFrame({
    'Nulos': df_raw.isnull().sum(),
    'Porcentaje': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'Tipo': df_raw.dtypes
})
display(nulos[nulos['Nulos'] > 0].sort_values('Porcentaje', ascending=False))

,Nulos,Porcentaje,Tipo
Nac.,30,100.00,float64
#,2,6.67,object
Valor de mercado,1,3.33,object


# ───── Decisiones de imputación — Plantel Huracán ─────
- Valor de mercado: 1 NaN (Nazareno Durán) — se deja como NaN. Transfermarkt no tiene dato disponible ('-' en el sitio original)
pandas ignora NaN en .mean(), el promedio del plantel no se ve afectado
- Nac.: si bien tiene más de 50% nulos, la dejo para imputarle datos manualmente

In [58]:

# Caso A — Separar texto con patrón conocido al final del string (nombre + posición)

# ── Definir los valores posibles de la parte a extraer ───────────────────
# Listar de más largo a más corto para que el regex priorice el match completo
valores_a_extraer = [
    'Mediocentro ofensivo',     # ← reemplazar con los valores del dataset
    'Lateral izquierdo',
    'Lateral derecho',
    'Interior derecho',
    'Interior izquierdo',
    'Extremo izquierdo',
    'Extremo derecho',
    'Delantero centro',
    'Defensa central',
    'Mediocentro',
    'Portero',
    'Pivote',
]

patron = '(' + '|'.join(valores_a_extraer) + ')'

# ── Extraer en columna nueva ──────────────────────────────────────────────
df_raw['posicion'] = df_raw['Jugadores'].str.extract(patron)             # ← reemplazar nombre columna origen

# ── Limpiar la columna original dejando solo el nombre ───────────────────
df_raw['jugador'] = df_raw['Jugadores'].str.replace(patron, '', regex=True).str.strip()

# ── Verificar que no quedaron NaN en posicion ────────────────────────────
print(f"NaN en posicion: {df_raw['posicion'].isnull().sum()}")
display(df_raw[['jugador', 'posicion']].head(10))

# ── Eliminar columna original una vez verificado ─────────────────────────
df_raw = df_raw.drop(columns=['Jugadores'])                              # ← reemplazar

NaN en posicion: 0


,jugador,posicion
0,Sebastián Meza,Portero
3,Hernán Galíndez,Portero
6,Nazareno Durán,Portero
9,Lucas Carrizo,Defensa central
12,Hugo Nervo,Defensa central
15,Mauro Villar,Defensa central
18,Nehuén Paz,Defensa central
21,Fabio Pereyra,Defensa central
24,Máximo Palazzo,Defensa central
27,Daniel Zabala,Defensa central


In [60]:

## ── Separar en dos columnas usando regex con grupos ──────────────────────
## Ejemplo: '14/03/2000 (26)' → fecha_nacimiento='14/03/2000', edad=26
#df_raw['fecha_nacimiento'] = df_raw['F. Nacim./Edad'].str.extract(r'(\d{2}/\d{2}/\d{4})')
#df_raw['edad'] = df_raw['F. Nacim./Edad'].str.extract(r'\((\d+)\)').astype(float)
#
## ── Convertir fecha a tipo datetime ──────────────────────────────────────
#df_raw['fecha_nacimiento'] = pd.to_datetime(df_raw['fecha_nacimiento'],
#                                             format='%d/%m/%Y',
#                                             errors='coerce')
#
## ── Verificar ────────────────────────────────────────────────────────────
#print(f"NaN en fecha_nacimiento: {df_raw['fecha_nacimiento'].isnull().sum()}")
#print(f"NaN en edad: {df_raw['edad'].isnull().sum()}")
#display(df_raw[['F. Nacim./Edad', 'fecha_nacimiento', 'edad']].head(5))
#
## ── Eliminar columna original una vez verificado ─────────────────────────
#df_raw = df_raw.drop(columns=['F. Nacim./Edad'])

# comento el bloque de código dado que ya se ejecutó y se borraron/agregaron columnas
display(df_raw)

,#,Nac.,Valor de mercado,posicion,jugador,fecha_nacimiento,edad
0,32,NaN,450 mil €,Portero,Sebastián Meza,2000-03-14,26.00
3,1,NaN,300 mil €,Portero,Hernán Galíndez,1987-03-30,39.00
6,27,NaN,NaN,Portero,Nazareno Durán,2004-05-10,22.00
9,3,NaN,800 mil €,Defensa central,Lucas Carrizo,1997-05-20,29.00
12,21,NaN,225 mil €,Defensa central,Hugo Nervo,1991-01-06,35.00
15,NaN,NaN,175 mil €,Defensa central,Mauro Villar,2001-01-11,25.00
18,30,NaN,175 mil €,Defensa central,Nehuén Paz,1993-04-28,33.00
21,6,NaN,150 mil €,Defensa central,Fabio Pereyra,1990-01-31,36.00
24,35,NaN,125 mil €,Defensa central,Máximo Palazzo,2005-06-06,21.00
27,22,NaN,100 mil €,Defensa central,Daniel Zabala,2003-01-17,23.00
